# Real-Time Credit Card Fraud Detection & Risk Intelligence Platform

## Notebook 04 — Feature Engineering

### Objective

The objective of this notebook is to transform the cleaned transaction data
into meaningful features that can help machine learning models identify
fraudulent transactions.

The features will capture:

- Transaction timing
- Customer spending behavior
- Merchant behavior
- Transaction amount patterns
- Customer age
- Geographic relationships between customer and merchant locations

These engineered features will be used in the machine learning stage.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.pyplot import yscale

fraudTrain = pd.read_csv('fraudTrain_cleaned.csv')
fraudTrain.head(1)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0


In [6]:
fraudTrain['trans_date_trans_time'] = pd.to_datetime(fraudTrain['trans_date_trans_time'])

In [7]:
fraudTrain['transaction_hour'] = (fraudTrain['trans_date_trans_time'].dt.hour)

In [8]:
fraudTrain['transaction_day_of_week'] = (
    fraudTrain['trans_date_trans_time'].dt.dayofweek
)

In [9]:
fraudTrain['transaction_month'] = (
    fraudTrain['trans_date_trans_time'].dt.month
)

In [11]:
fraudTrain['transaction_day'] = (
    fraudTrain['trans_date_trans_time'].dt.day
)

In [12]:
fraudTrain[
    [
        'trans_date_trans_time',
        'transaction_hour',
        'transaction_day_of_week',
        'transaction_month',
        'transaction_day'
    ]
].head()

,trans_date_trans_time,transaction_hour,transaction_day_of_week,transaction_month,transaction_day
0,2019-01-01 00:00:18,0,1,1,1
1,2019-01-01 00:00:44,0,1,1,1
2,2019-01-01 00:00:51,0,1,1,1
3,2019-01-01 00:01:16,0,1,1,1
4,2019-01-01 00:03:06,0,1,1,1


In [13]:
customer_avg_amount = (
    fraudTrain.groupby('cc_num')['amt']
    .transform('mean')
)

In [16]:
customer_avg_amount

0           87.393215
1           53.949320
2           65.870040
3           72.776673
4           95.178091
              ...    
1296670     63.182274
1296671    101.150621
1296672     65.235995
1296673     95.753691
1296674     68.929193
Name: amt, Length: 1296675, dtype: float64

In [17]:
fraudTrain['customer_avg_amount'] = customer_avg_amount

In [20]:
fraudTrain[
    ['cc_num', 'amt', 'customer_avg_amount']
].head(10)

,cc_num,amt,customer_avg_amount
0,2703186189652095,4.97,87.393215
1,630423337322,107.23,53.949320
2,38859492057661,220.11,65.870040
3,3534093764340240,45.00,72.776673
4,375534208663984,41.96,95.178091
5,4767265376804500,94.63,65.401685
6,30074693890476,44.54,90.289835
7,6011360759745864,71.65,68.635163
8,4922710831011201,4.27,68.591883
9,2720830304681674,198.39,94.141775


In [21]:
fraudTrain['log_amount'] = np.log1p(fraudTrain['amt'])

In [22]:
fraudTrain[['amt', 'log_amount']].head()

,amt,log_amount
0,4.97,1.786747
1,107.23,4.684259
2,220.11,5.398660
3,45.00,3.828641
4,41.96,3.760269


In [23]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [24]:
fraudTrain['distance_to_merchant'] = haversine_distance(
    fraudTrain['lat'],
    fraudTrain['long'],
    fraudTrain['merch_lat'],
    fraudTrain['merch_long']
)

In [25]:
fraudTrain[
    ['lat', 'long', 'merch_lat', 'merch_long', 'distance_to_merchant']
].head()

,lat,long,merch_lat,merch_long,distance_to_merchant
0,36.0788,-81.1781,36.011293,-82.048315,78.597568
1,48.8878,-118.2105,49.159047,-118.186462,30.212176
2,42.1808,-112.2620,43.150704,-112.154481,108.206083
3,46.2306,-112.1138,47.034331,-112.561071,95.673231
4,38.4207,-79.4629,38.674999,-78.632459,77.556744


In [26]:
fraudTrain['distance_to_merchant'].describe()

count    1.296675e+06
mean     7.611465e+01
std      2.911693e+01
min      2.225452e-02
25%      5.533491e+01
50%      7.823175e+01
75%      9.850327e+01
max      1.521172e+02
Name: distance_to_merchant, dtype: float64

In [27]:
categorical_columns = fraudTrain.select_dtypes(
    include='object'
).columns

categorical_columns

Index(['merchant', 'category', 'first', 'last', 'gender', 'street', 'city',
       'state', 'job', 'dob', 'trans_num'],
      dtype='object')

In [28]:
fraudTrain = pd.get_dummies(
    fraudTrain,
    columns=['category', 'gender', 'state'],
    drop_first=True,
    dtype=int
)

In [29]:
fraudTrain.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'amt', 'first', 'last',
       'street', 'city', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob',
       'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud',
       'transaction_hour', 'transaction_day_of_week', 'transaction_month',
       'transaction_day', 'customer_avg_amount', 'log_amount',
       'distance_to_merchant', 'category_food_dining',
       'category_gas_transport', 'category_grocery_net',
       'category_grocery_pos', 'category_health_fitness', 'category_home',
       'category_kids_pets', 'category_misc_net', 'category_misc_pos',
       'category_personal_care', 'category_shopping_net',
       'category_shopping_pos', 'category_travel', 'gender_M', 'state_AL',
       'state_AR', 'state_AZ', 'state_CA', 'state_CO', 'state_CT', 'state_DC',
       'state_DE', 'state_FL', 'state_GA', 'state_HI', 'state_IA', 'state_ID',
       'state_IL', 'state_IN', 'state_KS', 'state_KY', 'state_LA', 'state_MA',
       'sta

In [30]:
fraudTrain.shape

(1296675, 90)

In [32]:
fraudTrain = fraudTrain.sort_values(
    'trans_date_trans_time'
).reset_index(drop=True)

In [33]:
fraudTrain['previous_transaction_count'] = (
    fraudTrain.groupby('cc_num').cumcount()
)

In [34]:
previous_sum = (
    fraudTrain.groupby('cc_num')['amt']
    .cumsum()
    - fraudTrain['amt']
)

In [35]:
fraudTrain['previous_avg_amount'] = (
    previous_sum /
    fraudTrain['previous_transaction_count'].replace(0, np.nan)
)

In [36]:
fraudTrain['amount_to_previous_avg'] = (
    fraudTrain['amt'] /
    fraudTrain['previous_avg_amount']
)

In [37]:
fraudTrain['hours_since_previous_transaction'] = (
    fraudTrain.groupby('cc_num')['trans_date_trans_time']
    .diff()
    .dt.total_seconds() / 3600
)

In [38]:
fraudTrain[
    [
        'cc_num',
        'trans_date_trans_time',
        'amt',
        'previous_transaction_count',
        'previous_avg_amount',
        'amount_to_previous_avg',
        'hours_since_previous_transaction'
    ]
].head(20)

,cc_num,trans_date_trans_time,amt,previous_transaction_count,previous_avg_amount,amount_to_previous_avg,hours_since_previous_transaction
0,2703186189652095,2019-01-01 00:00:18,4.97,0,NaN,NaN,NaN
1,630423337322,2019-01-01 00:00:44,107.23,0,NaN,NaN,NaN
2,38859492057661,2019-01-01 00:00:51,220.11,0,NaN,NaN,NaN
3,3534093764340240,2019-01-01 00:01:16,45.00,0,NaN,NaN,NaN
4,375534208663984,2019-01-01 00:03:06,41.96,0,NaN,NaN,NaN
5,4767265376804500,2019-01-01 00:04:08,94.63,0,NaN,NaN,NaN
6,30074693890476,2019-01-01 00:04:42,44.54,0,NaN,NaN,NaN
7,6011360759745864,2019-01-01 00:05:08,71.65,0,NaN,NaN,NaN
8,4922710831011201,2019-01-01 00:05:18,4.27,0,NaN,NaN,NaN
9,2720830304681674,2019-01-01 00:06:01,198.39,0,NaN,NaN,NaN


In [39]:
fraudTrain['dob'] = pd.to_datetime(fraudTrain['dob'])

In [40]:
fraudTrain['customer_age'] = (
    (fraudTrain['trans_date_trans_time'] - fraudTrain['dob'])
    .dt.days / 365.25
).astype(int)

In [41]:
fraudTrain[['dob', 'trans_date_trans_time', 'customer_age']].head()

,dob,trans_date_trans_time,customer_age
0,1988-03-09,2019-01-01 00:00:18,30
1,1978-06-21,2019-01-01 00:00:44,40
2,1962-01-19,2019-01-01 00:00:51,56
3,1967-01-12,2019-01-01 00:01:16,51
4,1986-03-28,2019-01-01 00:03:06,32


In [42]:
columns_to_drop = [
    'trans_num',
    'first',
    'last',
    'street',
    'city',
    'merchant',
    'job',
    'dob',
    'trans_date_trans_time',
    'unix_time',
    'lat',
    'long',
    'merch_lat',
    'merch_long'
]

fraudTrain = fraudTrain.drop(columns=columns_to_drop)

In [43]:
fraudTrain.shape

(1296675, 81)

In [44]:
fraudTrain.isnull().sum()

cc_num                                0
amt                                   0
zip                                   0
city_pop                              0
is_fraud                              0
                                   ... 
previous_transaction_count            0
previous_avg_amount                 983
amount_to_previous_avg              983
hours_since_previous_transaction    983
customer_age                          0
Length: 81, dtype: int64

In [46]:
fraudTrain.isnull().sum().sum()

np.int64(0)

In [45]:
fraudTrain[
    [
        'previous_avg_amount',
        'amount_to_previous_avg',
        'hours_since_previous_transaction'
    ]
] = fraudTrain[
    [
        'previous_avg_amount',
        'amount_to_previous_avg',
        'hours_since_previous_transaction'
    ]
].fillna(0)

In [47]:
fraudTrain.to_csv(
    "fraudTrain_feature_engineered.csv",
    index=False
)

In [48]:
import os

os.path.exists("fraudTrain_feature_engineered.csv")

True